
# 🌍 Lab 3: การเขียนฟังก์ชัน
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจแนวคิดเกี่ยวกับ **ฟังก์ชัน (Function)** และข้อดีของการใช้ฟังก์ชัน
2. สามารถเขียนฟังก์ชันที่มีพารามิเตอร์และค่าที่คืนกลับ (Return Value) ได้
3. ใช้ฟังก์ชันในการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น การคำนวณระยะทางระหว่างจุด, การแปลงพิกัด, และการวิเคราะห์ค่า NDVI
4. สามารถใช้ **Lambda Function** และ **Recursive Function** ในบริบทของภูมิศาสตร์ได้

---

## 🔹 ตัวอย่างที่ 1: ฟังก์ชันคำนวณระยะทางระหว่างสองพิกัด


In [7]:

import math

def haversine(lat1, lon1, lat2, lon2):
    """ คำนวณระยะทางระหว่างสองพิกัดโดยใช้ Haversine Formula """
    R = 6371  # รัศมีโลก (กิโลเมตร)

    # แปลงองศาเป็นเรเดียน
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # คำนวณค่าความแตกต่าง
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # คำนวณระยะทางตาม Haversine Formula
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    distance = R * c

    return distance

# ทดสอบฟังก์ชัน
distance_km = haversine(13.7563, 100.5018, 18.7883, 98.9853)
print(f"ระยะทางระหว่างกรุงเทพฯ กับเชียงใหม่: {distance_km:.2f} กิโลเมตร")


ระยะทางระหว่างกรุงเทพฯ กับเชียงใหม่: 582.46 กิโลเมตร



## 🔹 ตัวอย่างที่ 2: ฟังก์ชันคำนวณค่า NDVI


In [2]:

def calculate_ndvi(nir, red):
    """ คำนวณค่า NDVI โดยใช้สูตร (NIR - RED) / (NIR + RED) """
    if nir + red == 0:
        return None  # หลีกเลี่ยงการหารด้วยศูนย์
    return (nir - red) / (nir + red)

# ทดสอบฟังก์ชัน
nir_value = 0.8
red_value = 0.3
ndvi = calculate_ndvi(nir_value, red_value)
print(f"ค่า NDVI: {ndvi:.2f}")


ค่า NDVI: 0.45



## 🔹 ตัวอย่างที่ 3: ใช้ Lambda Function คำนวณ NDVI


In [15]:

ndvi_lambda = lambda nir, red: (nir - red) / (nir + red) if (nir + red) != 0 else None

# ทดสอบ
print(f"ค่า NDVI (Lambda): {ndvi_lambda(0.8, 0.3):.2f}")


ค่า NDVI (Lambda): 0.45



## 🔹 ตัวอย่างที่ 4: ฟังก์ชันแบบ Recursive หาค่า Factorial


In [4]:

def factorial(n):
    """ คำนวณค่า factorial โดยใช้ recursive function """
    if n == 0 or n == 1:
        return 1
    return n * factorial(n - 1)

# ทดสอบ
print(f"5! = {factorial(5)}")


5! = 120



# 📝 **กิจกรรมในแลป**

1. **แบบฝึกหัด 1**: สร้างฟังก์ชัน `convert_to_utm(lat, lon)` เพื่อแปลงค่าพิกัดจาก Lat/Lon เป็น UTM
2. **แบบฝึกหัด 2**: เขียนฟังก์ชัน `filter_locations_by_lat(locations, min_lat)` เพื่อตรวจสอบว่าจังหวัดไหนอยู่เหนือค่าละติจูดที่กำหนด
3. **แบบฝึกหัด 3**: ใช้ **Lambda Function** คำนวณระยะทางระหว่างสองจุด
4. **แบบฝึกหัด 4**: ใช้ **Recursive Function** คำนวณลำดับฟีโบนัชชี (`Fibonacci`)


**แบบฝึกหัดที่ 1:** สร้างฟังก์ชันเพื่อแปลงค่าพิกัดจาก Lat/Lon เป็น UTM

In [8]:
import sys
!{sys.executable} -m pip install pyproj

In [13]:
from pyproj import Transformer

def convert_to_utm(lat, lon):
    """
    แปลงพิกัด Lat/Lon (WGS84) เป็น UTM โดยการคำนวณโซน UTM อย่างชัดเจน
    """
    # คำนวณ UTM zone จากลองจิจูด
    utm_zone_number = int((lon + 180) / 6) + 1

    # กำหนด Hemisphere: N (North) สำหรับพิกัดละติจูด >= 0, S (South) สำหรับละติจูด < 0
    hemisphere = 'N' if lat >= 0 else 'S'

    # สร้าง EPSG code สำหรับ UTM zone นั้นๆ
    # สำหรับ WGS84, โซน UTM เหนือเส้นศูนย์สูตรจะเริ่มต้นที่ 326xx, ใต้เส้นศูนย์สูตรจะเริ่มต้นที่ 327xx
    if hemisphere == 'N':
        utm_epsg = 32600 + utm_zone_number
    else:
        utm_epsg = 32700 + utm_zone_number

    # สร้าง Transformer เพื่อแปลงจาก WGS84 (EPSG:4326) ไปยัง UTM ที่กำหนด
    transformer = Transformer.from_crs("epsg:4326", f"epsg:{utm_epsg}", always_xy=True)

    # ทำการแปลงพิกัด
    easting, northing = transformer.transform(lon, lat)

    return easting, northing, f"UTM Zone {utm_zone_number}{hemisphere}"

# ทดสอบฟังก์ชัน
lat_bangkok, lon_bangkok = 13.7563, 100.5018 # กรุงเทพฯ
lat_chiangmai, lon_chiangmai = 18.7883, 98.9853 # เชียงใหม่
lat_lopburi, lon_lopburi = 14.7995, 100.6533 # ลพบุรี

# แปลงพิกัดกรุงเทพฯ
easting_bkk, northing_bkk, utm_zone_bkk = convert_to_utm(lat_bangkok, lon_bangkok)
print(f"พิกัดกรุงเทพฯ (Lat: {lat_bangkok}, Lon: {lon_bangkok}) ในระบบ UTM: E {easting_bkk:.2f}, N {northing_bkk:.2f}, {utm_zone_bkk}")

# แปลงพิกัดเชียงใหม่
easting_cm, northing_cm, utm_zone_cm = convert_to_utm(lat_chiangmai, lon_chiangmai)
print(f"พิกัดเชียงใหม่ (Lat: {lat_chiangmai}, Lon: {lon_chiangmai}) ในระบบ UTM: E {easting_cm:.2f}, N {northing_cm:.2f}, {utm_zone_cm}")

# แปลงพิกัดลพบุรี
easting_lbr, northing_lbr,  utm_zone_lbr = convert_to_utm(lat_lopburi, lon_lopburi)
print(f"พิกัดลพบุรี (lat: {lat_lopburi}, lon:{lon_lopburi}) ในระบบ UTM: E {easting_lbr: .2f}, N {northing_lbr: .2f}, {utm_zone_lbr}")

# แปลงพิกัดภูเก็ต
lat_phuket, lon_phuket = 7.8731, 98.3924
easting_pkt, northing_pkt, utm_zone_pkt = convert_to_utm(lat_phuket, lon_phuket)
print(f"พิกัดภูเก็ต (lat: {lat_phuket}, lon:{lon_phuket}) ในระบบ UTM: E {easting_pkt: .2f}, N {northing_pkt: .2f}, {utm_zone_pkt}")


พิกัดกรุงเทพฯ (Lat: 13.7563, Lon: 100.5018) ในระบบ UTM: E 662366.60, N 1521280.66, UTM Zone 47N
พิกัดเชียงใหม่ (Lat: 18.7883, Lon: 98.9853) ในระบบ UTM: E 498450.88, N 2077403.64, UTM Zone 47N
พิกัดลพบุรี (lat: 14.7995, lon:100.6533) ในระบบ UTM: E  677928.25, N  1636805.87, UTM Zone 47N
พิกัดภูเก็ต (lat: 7.8731, lon:98.3924) ในระบบ UTM: E  433021.22, N  870317.55, UTM Zone 47N


**แบบฝึกหัดที่ 2:** เขียนฟังก์ชันเพื่อตรวจสอบว่าจังหวัดไหนอยู่เหนือค่าละติจูดที่กำหนด

In [14]:
# สร้างข้อมูลตัวอย่างของจังหวัดและละติจูด
locations_data = [
    {"name": "กรุงเทพมหานคร", "latitude": 13.7563, "longitude": 100.5018},
    {"name": "เชียงใหม่", "latitude": 18.7883, "longitude": 98.9853},
    {"name": "ภูเก็ต", "latitude": 7.8804, "longitude": 98.3924},
    {"name": "ลพบุรี", "latitude": 14.7995, "longitude": 100.6533},
    {"name": "สงขลา", "latitude": 7.2023, "longitude": 100.5982},
    {"name": "ขอนแก่น", "latitude": 16.4328, "longitude": 102.8335}
]

def filter_locations_by_lat(locations, min_lat):
    """
    คัดกรองรายการสถานที่ (locations) เพื่อหาว่าสถานที่ใดมีค่าละติจูดมากกว่าหรือเท่ากับ min_lat ที่กำหนด

    Parameters:
    locations (list): รายการของ dictionary แต่ละ dictionary ควรมีคีย์ 'name' และ 'latitude'
    min_lat (float): ค่าละติจูดขั้นต่ำที่ต้องการคัดกรอง

    Returns:
    list: รายการของสถานที่ที่ผ่านการคัดกรอง
    """
    filtered_locations = []
    for location in locations:
        if location['latitude'] >= min_lat:
            filtered_locations.append(location)
    return filtered_locations

# ทดสอบฟังก์ชัน
min_latitude_to_filter = 15.0 # กำหนดค่าละติจูดขั้นต่ำที่ 15 องศาเหนือ
locations_above_min_lat = filter_locations_by_lat(locations_data, min_latitude_to_filter)

print(f"จังหวัดที่อยู่เหนือละติจูด {min_latitude_to_filter} องศาเหนือ:")
if locations_above_min_lat:
    for loc in locations_above_min_lat:
        print(f"- {loc['name']} (ละติจูด: {loc['latitude']})")
else:
    print("ไม่มีจังหวัดใดที่อยู่เหนือละติจูดที่กำหนด")


# ทดสอบอีกครั้งด้วยค่าละติจูดที่ต่ำลงมา
min_latitude_to_filter_2 = 10.0 # กำหนดค่าละติจูดขั้นต่ำที่ 10 องศาเหนือ
locations_above_min_lat_2 = filter_locations_by_lat(locations_data, min_latitude_to_filter_2)

print(f"\nจังหวัดที่อยู่เหนือละติจูด {min_latitude_to_filter_2} องศาเหนือ:")
if locations_above_min_lat_2:
    for loc in locations_above_min_lat_2:
        print(f"- {loc['name']} (ละติจูด: {loc['latitude']})")
else:
    print("ไม่มีจังหวัดใดที่อยู่เหนือละติจูดที่กำหนด")

จังหวัดที่อยู่เหนือละติจูด 15.0 องศาเหนือ:
- เชียงใหม่ (ละติจูด: 18.7883)
- ขอนแก่น (ละติจูด: 16.4328)

จังหวัดที่อยู่เหนือละติจูด 10.0 องศาเหนือ:
- กรุงเทพมหานคร (ละติจูด: 13.7563)
- เชียงใหม่ (ละติจูด: 18.7883)
- ลพบุรี (ละติจูด: 14.7995)
- ขอนแก่น (ละติจูด: 16.4328)


**แบบฝึกหัดที่ 3:** ใช้ Lambda Function คำนวณระยะทางระหว่างสองจุด

In [18]:
import math

haversine_lambda = lambda lat1, lon1, lat2, lon2: (6371 * 2 * math.atan2(math.sqrt(math.sin((math.radians(lat2) - math.radians(lat1))/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin((math.radians(lon2) - math.radians(lon1))/2)**2), math.sqrt(1 - (math.sin((math.radians(lat2) - math.radians(lat1))/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin((math.radians(lon2) - math.radians(lon1))/2)**2))))

# ทดสอบ Lambda Function
lat_bangkok, lon_bangkok = 13.7563, 100.5018 # กรุงเทพฯ
lat_chiangmai, lon_chiangmai = 18.7883, 98.9853 # เชียงใหม่
lat_phuket, lon_phuket = 7.8731, 98.3924 # ภูเก็ต
lat_lopburi, lon_lopburi = 14.7995, 100.6533 # ลพบุรี

distance_bkk_cm = haversine_lambda(lat_bangkok, lon_bangkok, lat_chiangmai, lon_chiangmai)
print(f"ระยะทางระหว่างกรุงเทพฯ กับเชียงใหม่ (Lambda): {distance_bkk_cm:.2f} กิโลเมตร")

distance_bkk_phuket = haversine_lambda(lat_bangkok, lon_bangkok, lat_phuket, lon_phuket)
print(f"ระยะทางระหว่างกรุงเทพฯ กับภูเก็ต (Lambda): {distance_bkk_phuket:.2f} กิโลเมตร")

distance_cm_lopburi = haversine_lambda(lat_chiangmai, lon_chiangmai, lat_lopburi, lon_lopburi)
print(f"ระยะทางระหว่างเชียงใหม่ กับลพบุรี (Lambda): {distance_cm_lopburi:.2f} กิโลเมตร")

ระยะทางระหว่างกรุงเทพฯ กับเชียงใหม่ (Lambda): 582.46 กิโลเมตร
ระยะทางระหว่างกรุงเทพฯ กับภูเก็ต (Lambda): 693.53 กิโลเมตร
ระยะทางระหว่างเชียงใหม่ กับลพบุรี (Lambda): 477.74 กิโลเมตร


**แบบฝึกหัดที่4:** ใช้ Recursive Function คำนวณลำดับฟีโบนัชชี

In [19]:
def fibonacci(n):
    """
    คำนวณลำดับฟีโบนัชชีตัวที่ n โดยใช้ recursive function
    """
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

# ทดสอบฟังก์ชัน
print(f"Fibonacci(0) = {fibonacci(0)}")
print(f"Fibonacci(1) = {fibonacci(1)}")
print(f"Fibonacci(5) = {fibonacci(5)}")
print(f"Fibonacci(10) = {fibonacci(10)}")

Fibonacci(0) = 0
Fibonacci(1) = 1
Fibonacci(5) = 5
Fibonacci(10) = 55
